<a href="https://colab.research.google.com/github/Ali-Hamza-developer/NLP/blob/main/08_bag_of_words.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 08_Bag of Words

## 1. What is Bag of Words (BOW)? (Easy Explanation)

Machine learning models only understand **numbers**, not raw text. Bag of Words is the simplest way to turn text into numbers.

**Idea:** Take every unique word across all your documents, and for each document just count how many times each word appears. Word ORDER is ignored — only word COUNT matters (that's why it's called a "bag", not a "sentence").

```
Doc 1: "I love this movie"
Doc 2: "I hate this movie"

Vocabulary: [I, love, hate, this, movie]

Doc 1 -> [1, 1, 0, 1, 1]
Doc 2 -> [1, 0, 1, 1, 1]
```

In sklearn, `CountVectorizer` builds this vocabulary and does the counting for you automatically.

---
# PART A: Tutorial — Spam Detection using Bag of Words

**Dataset needed:** `spam.csv` (SMS messages labeled spam/ham) — place it in the same folder as this notebook.

In [18]:
import pandas as pd
import numpy as np


In [19]:
df = pd.read_csv("spam.csv")
df.head()


,Category,Message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [20]:
df.Category.value_counts()   # check how many spam vs ham messages -> good to know if data is balanced


,count
Category,
ham,4825
spam,747


In [21]:
# convert text labels into 0/1 numbers so the model can use them
df["spam"] = df["Category"].apply(lambda x: 1 if x == "spam" else 0)


In [22]:
df.shape


(5572, 3)

In [23]:
df.head()


,Category,Message,spam
0,ham,"Go until jurong point, crazy.. Available only ...",0
1,ham,Ok lar... Joking wif u oni...,0
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,1
3,ham,U dun say so early hor... U c already then say...,0
4,ham,"Nah I don't think he goes to usf, he lives aro...",0


## Train-Test Split

Always split your data BEFORE vectorizing, so the test set stays truly unseen.

In [24]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(df.Message, df.spam, test_size=0.2)


In [25]:
X_train.shape


(4457,)

In [26]:
X_test.shape


(1115,)

## Creating Bag of Words with `CountVectorizer`

In [27]:
from sklearn.feature_extraction.text import CountVectorizer

v = CountVectorizer()

X_train_cv = v.fit_transform(X_train.values)   # fit_transform: LEARNS vocabulary AND converts text to numbers
X_train_cv


<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 58983 stored elements and shape (4457, 7726)>

**Output:** a sparse matrix like `<4457x7726 sparse matrix ...>` — rows = messages, columns = unique words in the vocabulary. Sparse because most word-counts per message are 0.

In [28]:
X_train_cv.toarray()[:2][0]   # convert to a normal array to actually see the numbers


array([0, 0, 0, ..., 0, 0, 0])

In [29]:
X_train_cv.shape   # (num_messages, vocabulary_size)


(4457, 7726)

In [30]:
v.get_feature_names_out()[1771]   # look up which word corresponds to column index 1771


'cheap'

In [31]:
v.vocabulary_   # full word -> column_index mapping


{'alright': 934,
 'if': 3608,
 'you': 7686,
 're': 5569,
 'sure': 6602,
 'let': 4088,
 'me': 4413,
 'know': 3968,
 'when': 7462,
 'leaving': 4067,
 'hey': 3426,
 'so': 6249,
 'this': 6833,
 'sat': 5889,
 'are': 1057,
 'we': 7397,
 'going': 3185,
 'for': 2953,
 'the': 6794,
 'intro': 3726,
 'pilates': 5208,
 'only': 4937,
 'or': 4964,
 'kickboxing': 3934,
 'too': 6955,
 'do': 2395,
 'thing': 6822,
 'change': 1745,
 'that': 6790,
 'sentence': 5984,
 'into': 3724,
 'because': 1295,
 'want': 7360,
 'concentrate': 1970,
 'in': 3650,
 'my': 4669,
 'educational': 2553,
 'career': 1679,
 'im': 3626,
 'here': 3416,
 'someone': 6266,
 'has': 3348,
 'asked': 1107,
 'our': 4996,
 'dating': 2194,
 'service': 5996,
 'contact': 2005,
 'cant': 1662,
 'guess': 3276,
 'who': 7477,
 'call': 1628,
 '09058091854': 168,
 'now': 4842,
 'all': 921,
 'will': 7497,
 'be': 1284,
 'revealed': 5746,
 'po': 5257,
 'box385': 1478,
 'm6': 4287,
 '6wu': 612,
 'pls': 5248,
 'clarify': 1850,
 'back': 1208,
 'an': 966,
 

In [32]:
X_train_np = X_train_cv.toarray()
X_train_np[0]   # word-count vector for the first training message


array([0, 0, 0, ..., 0, 0, 0])

In [33]:
np.where(X_train_np[0] != 0)   # which columns (words) are non-zero for this message


(array([ 934, 3608, 3968, 4067, 4088, 4413, 5569, 6602, 7462, 7686]),)

In [34]:
X_train_np[0][1771]   # how many times the word at index 1771 appears in message 0


np.int64(0)

## Training a Naive Bayes Model

`MultinomialNB` is a classic, fast, and surprisingly strong choice for text classification with Bag of Words.

In [35]:
from sklearn.naive_bayes import MultinomialNB

model = MultinomialNB()
model.fit(X_train_cv, y_train)


MultinomialNB()

In [36]:
X_test_cv = v.transform(X_test)   # transform only (NOT fit_transform) -> reuse the SAME vocabulary learned from train


## Evaluating the Model

In [37]:
from sklearn.metrics import classification_report

y_pred = model.predict(X_test_cv)

print(classification_report(y_test, y_pred))


              precision    recall  f1-score   support

           0       0.99      1.00      0.99       974
           1       0.97      0.94      0.95       141

    accuracy                           0.99      1115
   macro avg       0.98      0.97      0.97      1115
weighted avg       0.99      0.99      0.99      1115



In [38]:
# quick test on brand new emails
emails = [
    "Hey mohan, can we get together to watch footbal game tomorrow?",
    "Upto 20% discount on parking, exclusive offer just for you. Dont miss this reward!"
]

emails_count = v.transform(emails)
model.predict(emails_count)


array([0, 1])

**Output:** `array([0, 1])` — first message predicted as ham (0), second as spam (1). Makes sense — the second one screams "discount / offer / reward", classic spam wording.

## Cleaner Way — Using `sklearn.Pipeline`

Instead of manually calling `fit_transform`, `transform`, then `fit`, `predict` separately, a `Pipeline` chains all the steps together, so you just call `.fit()` and `.predict()` once.

In [39]:
from sklearn.pipeline import Pipeline

clf = Pipeline([
    ("vectorizer", CountVectorizer()),
    ("nb", MultinomialNB())
])


In [40]:
clf.fit(X_train, y_train)   # pipeline handles vectorizing + training internally


Pipeline(steps=[('vectorizer', CountVectorizer()), ('nb', MultinomialNB())])

In [41]:
y_pred = clf.predict(X_test)

print(classification_report(y_test, y_pred))


              precision    recall  f1-score   support

           0       0.99      1.00      0.99       974
           1       0.97      0.94      0.95       141

    accuracy                           0.99      1115
   macro avg       0.98      0.97      0.97      1115
weighted avg       0.99      0.99      0.99      1115



Same result as before, but far less code — and no risk of accidentally mismatching train/test vocabularies, since the pipeline handles that automatically.

---
## Quick Cheat Sheet

| Task | Code |
|---|---|
| Create vectorizer | `CountVectorizer()` |
| Learn vocab + convert (train) | `v.fit_transform(X_train)` |
| Convert only, same vocab (test) | `v.transform(X_test)` |
| See vocabulary | `v.vocabulary_` / `v.get_feature_names_out()` |
| Build a pipeline | `Pipeline([("vectorizer", CountVectorizer()), ("model", SomeClassifier())])` |
| Evaluate | `classification_report(y_test, y_pred)` |

---

---
# PART B: Exercises — Movie Review Sentiment Classification (with Solutions)

**Dataset needed:** a file with two columns: `review` and `sentiment` (positive/negative).

> If you have the real dataset (`movies_sentiment_data.csv` from [Kaggle's IMDB 50K reviews](https://www.kaggle.com/datasets/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews)), just put it in the same folder and skip the next cell — `pd.read_csv("movies_sentiment_data.csv")` will pick it up directly.
>
> **Don't have the file?** Run the next cell — it generates a synthetic movie-review dataset locally and saves it as `movies_sentiment_data.csv`, so everything below runs exactly the same either way.

You'll build 3 pipelines (Random Forest, KNN, Multinomial Naive Bayes) using Bag of Words, and compare them.

In [42]:
import os
import random

# Only generate a synthetic dataset if the real file isn't already present
if not os.path.exists("movies_sentiment_data.csv"):

    random.seed(42)

    positive_templates = [
        "This movie was absolutely brilliant, I loved every minute of it.",
        "A masterpiece with stunning visuals and a gripping story.",
        "The acting was superb and the plot kept me hooked till the end.",
        "One of the best films I have seen this year, highly recommended.",
        "Beautifully directed, emotionally powerful, and truly inspiring.",
        "Great chemistry between the leads made this movie a joy to watch.",
        "The soundtrack and cinematography were outstanding throughout.",
        "A heartwarming story with brilliant performances from the whole cast.",
        "Funny, clever, and surprisingly touching all at the same time.",
        "I was on the edge of my seat, this thriller delivers non-stop excitement."
    ]

    negative_templates = [
        "This movie was a complete waste of time and money.",
        "The plot made no sense and the acting felt wooden.",
        "Boring from start to finish, I almost fell asleep.",
        "Terrible dialogue and a predictable, dull storyline.",
        "I regret watching this, it was painfully slow and dragged on.",
        "The special effects were cheap and the story was forgettable.",
        "None of the characters were likable, a real disappointment.",
        "Poorly written and badly directed, avoid this one.",
        "The pacing was awful and the ending felt rushed and lazy.",
        "A disaster of a film with weak performances all around."
    ]

    movie_words = ["film", "movie", "picture", "flick", "production", "story", "drama", "feature"]

    reviews = []
    sentiments = []

    # build ~300 reviews by mixing templates with small variations
    for _ in range(150):
        pos = random.choice(positive_templates)
        neg = random.choice(negative_templates)
        extra = random.choice(movie_words)
        reviews.append(pos + f" Overall a fantastic {extra}.")
        sentiments.append("positive")
        reviews.append(neg + f" Overall a disappointing {extra}.")
        sentiments.append("negative")

    df_synthetic = pd.DataFrame({"review": reviews, "sentiment": sentiments})
    df_synthetic.to_csv("movies_sentiment_data.csv", index=False)

    print("Synthetic movies_sentiment_data.csv created with", len(df_synthetic), "rows")
else:
    print("movies_sentiment_data.csv already exists, using the real file")


Synthetic movies_sentiment_data.csv created with 300 rows


> **Note on the synthetic data:** these reviews are built from a small set of templates, so the vocabulary is repetitive and the classes are very easy to separate — you'll likely see much higher (near-perfect) scores here than on messy real-world reviews. The *code and workflow* are identical either way; swap in the real Kaggle CSV any time for realistic numbers.

In [43]:
# import necessary libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report


In [44]:
# 1. read the data
df = pd.read_csv("movies_sentiment_data.csv")

# 2. shape of the data
print(df.shape)

# 3. top 5 rows
df.head()


(300, 2)


,review,sentiment
0,A masterpiece with stunning visuals and a grip...,positive
1,This movie was a complete waste of time and mo...,negative
2,"One of the best films I have seen this year, h...",positive
3,"Terrible dialogue and a predictable, dull stor...",negative
4,A masterpiece with stunning visuals and a grip...,positive


In [45]:
# create a new column "Category": 1 = positive, 0 = negative
df["Category"] = df["sentiment"].apply(lambda x: 1 if x == "positive" else 0)
df.head()


,review,sentiment,Category
0,A masterpiece with stunning visuals and a grip...,positive,1
1,This movie was a complete waste of time and mo...,negative,0
2,"One of the best films I have seen this year, h...",positive,1
3,"Terrible dialogue and a predictable, dull stor...",negative,0
4,A masterpiece with stunning visuals and a grip...,positive,1


In [46]:
# check whether the target labels are balanced
df.Category.value_counts()


,count
Category,
1,150
0,150


In [47]:
# train-test split, 20% test size
X_train, X_test, y_train, y_test = train_test_split(
    df.review, df.Category, test_size=0.2, random_state=2022, stratify=df.Category
)

print(X_train.shape, X_test.shape)


(240,) (60,)


## Exercise 1 — Random Forest (n_estimators=50, criterion='entropy')

In [48]:
# 1. create pipeline
clf_rf = Pipeline([
    ("vectorizer", CountVectorizer()),
    ("rf", RandomForestClassifier(n_estimators=50, criterion="entropy"))
])

# 2. fit
clf_rf.fit(X_train, y_train)

# 3. predict
y_pred = clf_rf.predict(X_test)

# 4. classification report
print(classification_report(y_test, y_pred))


              precision    recall  f1-score   support

           0       1.00      1.00      1.00        30
           1       1.00      1.00      1.00        30

    accuracy                           1.00        60
   macro avg       1.00      1.00      1.00        60
weighted avg       1.00      1.00      1.00        60



## Exercise 2 — KNN (n_neighbors=10, metric='euclidean')

In [49]:
# 1. create pipeline
clf_knn = Pipeline([
    ("vectorizer", CountVectorizer()),
    ("knn", KNeighborsClassifier(n_neighbors=10, metric="euclidean"))
])

# 2. fit
clf_knn.fit(X_train, y_train)

# 3. predict
y_pred = clf_knn.predict(X_test)

# 4. classification report
print(classification_report(y_test, y_pred))


              precision    recall  f1-score   support

           0       1.00      1.00      1.00        30
           1       1.00      1.00      1.00        30

    accuracy                           1.00        60
   macro avg       1.00      1.00      1.00        60
weighted avg       1.00      1.00      1.00        60



## Exercise 3 — Multinomial Naive Bayes

In [50]:
# 1. create pipeline
clf_nb = Pipeline([
    ("vectorizer", CountVectorizer()),
    ("nb", MultinomialNB())
])

# 2. fit
clf_nb.fit(X_train, y_train)

# 3. predict
y_pred = clf_nb.predict(X_test)

# 4. classification report
print(classification_report(y_test, y_pred))


              precision    recall  f1-score   support

           0       1.00      1.00      1.00        30
           1       1.00      1.00      1.00        30

    accuracy                           1.00        60
   macro avg       1.00      1.00      1.00        60
weighted avg       1.00      1.00      1.00        60



## Why does KNN perform worse than Random Forest and Multinomial NB here?

**Answer:**

1. **Very high dimensions, mostly zeros (curse of dimensionality).** Bag of Words with `CountVectorizer` on movie reviews creates thousands of columns (one per unique word), but each review only uses a tiny fraction of them. In such high-dimensional, sparse spaces, the notion of "distance" that KNN depends on becomes unreliable — almost every pair of points ends up looking roughly equally "far" from each other.

2. **KNN treats every word/dimension equally.** It has no concept of which words are actually meaningful (like "boring", "terrible", "brilliant") versus common filler words. Random Forest and Naive Bayes both implicitly learn which features matter more for the prediction; KNN's raw Euclidean distance does not.

3. **Naive Bayes is actually built for this exact setup.** Multinomial NB was specifically designed for word-count data — it directly models word frequency per class, so it thrives on Bag of Words features.

4. **Random Forest handles high-dimensional sparse data reasonably well** because it builds decision trees that pick a few useful word-columns to split on at each step, rather than relying on distance across all dimensions at once.

**Takeaway:** For text/Bag-of-Words data, distance-based models like KNN generally underperform compared to Naive Bayes or tree-based ensembles like Random Forest.